In [ ]:
!pip install -q streamlit
!pip install -q langchain
!pip install -q langchain-community
!pip install -q langchain-google-genai
!pip install -q langchain-text-splitters
!pip install -q chromadb
!pip install -q pypdf

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.2/9.2 MB 31.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 32.6 MB/s eta 0:00:00


In [ ]:
from google.colab import userdata
import os

os.environ["GOOGLE_API_KEY"] = userdata.get("GOOGLE_API_KEY")

In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI

llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    temperature=0.2
)

print(llm.invoke("Hello").content)

Hello! How can I help you today?


In [ ]:
from google.colab import files

uploaded = files.upload()

pdf_file = list(uploaded.keys())[0]

Saving Pavithra_Resume.pdf (4).pdf to Pavithra_Resume.pdf (4) (1).pdf


In [ ]:
from langchain_community.document_loaders import PyPDFLoader

loader = PyPDFLoader(pdf_file)

documents = loader.load()

print(len(documents))

1


In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)

docs = splitter.split_documents(documents)

print(len(docs))

4


In [ ]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings

embeddings = GoogleGenerativeAIEmbeddings(
    model="models/gemini-embedding-001"
)

In [ ]:
from langchain_community.vectorstores import Chroma

vectorstore = Chroma.from_documents(
    docs,
    embeddings
)

retriever = vectorstore.as_retriever()

In [ ]:
while True:

    query = input("Ask: ")

    if query.lower() == "exit":
        break

    docs_found = retriever.invoke(query)

    context = "\n\n".join(
        [doc.page_content for doc in docs_found[:4]]
    )

    prompt = f"""
    Answer using the context below.

    Context:
    {context}

    Question:
    {query}
    """

    response = llm.invoke(prompt)

    print(response.content)

Ask: Rewrite my resume summary professionally
Highly motivated Artificial Intelligence & Data Science undergraduate with hands-on experience in Machine Learning, Data Analytics, and Cloud Computing, gained through internships and projects. Proficient in Python, SQL, Power BI, Tableau, and Scikit-Learn. Eager to apply AI and data-driven solutions to real-world business and technical challenges.
Ask: exit
